# From a Gaussian measurement to four informative bins

Suppose each event is one measurement $x\sim\mathcal N(\mu,1)$ and the downstream task is to estimate the unknown location $\mu$. We deliberately keep only four hard-bin counts. This notebook derives the score, explores the data, calls the public `fit_scores` API, and measures what the compression retained.

## The score is the relevant coordinate

At the reference point $\mu_0=0$, $\partial_\mu\log p(x\mid\mu)|_{\mu_0}=x$. The observation and its score happen to be identical here. That makes this a useful analytic check: grouping nearby score values should preserve information about $\mu$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fisherbin as fb
from examples.synthetic_problems import gaussian_location

problem = gaussian_location()
train, validation, test = problem.train, problem.validation, problem.test
train.observations.shape, validation.observations.shape, test.observations.shape

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(train.observations[:, 0], bins=40, density=True, alpha=0.75)
ax.set(
    xlabel="measurement x = score s(x)",
    ylabel="density",
    title="Training events at the reference point",
);

## Learn the hard partition

`fit_scores` consumes one row per event and one column per score direction. Validation rows are diagnostic only: they do not influence gradients, stopping, or checkpoint selection.

In [ ]:
partition = fb.fit_scores(
    train.scores,
    weights=train.weights,
    n_bins=4,
    validation_scores=validation.scores,
    validation_weights=validation.weights,
    config=fb.KMeansConfig(seed=42, n_init=4),
)
labels = np.asarray(partition.predict(test.scores))
counts = np.bincount(labels, minlength=partition.n_bins)
test_report = partition.evaluate(test.scores, test.weights)
counts, test_report.geometric_mean_retention

In [ ]:
order = np.argsort(test.observations[:, 0])
fig, ax = plt.subplots(figsize=(8, 3))
ax.scatter(test.observations[order, 0], labels[order], c=labels[order], cmap="tab10", s=5)
ax.set(
    xlabel="measurement x",
    ylabel="hard-bin label",
    yticks=range(4),
    title="The learned one-dimensional partition",
);

## What leaves the library

The fitted object maps future events to stable integer labels. FisherBin reports compression information, but it does not estimate $\mu$ from the four counts; the downstream likelihood remains application code. Increasing the number of bins approaches the unbinned information, while a finite histogram necessarily loses some.